1. **Nome(s):** Leonardo Silveira Kiepper

2. **Matrícula(s):** 2026140417

3. **E-mail(s) institucional(is):** leonardo.kiepper@edu.ufes.br

3. **Curso(s):** Doutorado em Ciência da Computação

4. **Link do seu Colab com suas soluções (para que o professor possa acessar e rodar o código):** https://colab.research.google.com/github/LeoKiepper/Ciencia-de-Dados/blob/main/trabalho1/T1_leonardo_kiepper.ipynb


# Introdução

Cada compra que uma pessoa faz gera dados que são úteis para diversas aplicações práticas. Por exemplo, em se tratando de artigos de mercearia ou supermercado, ter um registro dos preços de produtos de interesse permite o monitoramento de gastos pessoais, precificação de receitas, etc. De modo geral, para tais aplicações serem viáveis em escala pessoal, é fundamental que a solução seja de baixo custo, de baixa manutenção, de baixo esforço e de baixa dificuldade técnica do lado do usuário. Isto elimina soluções como controle de estoque informatizado, que muitos supermercados usam.

Por exigência de lei, quando requisitado, estabelecimentos já entregam o cupom fiscal detalhando a compra. Sendo assim, o cupom fiscal é o artefato físico natural de onde extrair os dados necessários para estas aplicaçãoes, dadas as restrições acima. No entanto, a tarefa de ler cupons fiscais e extrair esses dados manualmente não é prática, devido ao volume de dados necessário para que a solução implementada tenha poder preditivo razoável.

De modo geral, os dados importantes para tais aplicações, que estariam presentes em uma nota ou cupom fiscal impresso, não são prontamente utilizáveis. Além da etapa de digitalização para converter em arquivo digital, e obrigatoriamente o reconhecimento óptico de caracteres (OCR) para converter imagem para texto, é necessário uma passada de extração de dados estruturada, na qual dados como descrição do produto e valor, são individualizados num arquivo tabular. Os dados mínimos de interesse são a descrição do produto, e o valor.

Mesmo tendo feito uma leitura e extração estruturada perfeitas, isto é, de modo que os dados constando no cupom fiscal físico tenham sido copiados exatamente para o arquivo digital, os dados ainda precisam ser curados, especificamente a descrição do produto. O desafio aqui é que a descrição do produto constando num cupom fiscal é extremamente truncada, para caber na dimensão limitada do papel térmico que normalmente é utilizado para esse propósito. Soma-se a isso os fatos de que cada comércio descreve e abrevia o produto de sua forma particular, e que, depois de identificado o produto a partir da descrição, é necessário adotar alguma forma padronizada de indexar o produto identificado para que a solução implementada seja amplamente aplicável a situações reais e possa ser generalizada para boa quantidade de casos particulares, e fica evidente que o problema não é trivial.

Limitando o escopo do projeto a predizer a descrição completa a partir da descrição truncada, procurou-se um conjunto de dados que suportasse essa tarefa. A base de dados aberta ELEVEN (https://github.com/vinidiol/descmerc) traz aproximadamente 1,1 milhão de linhas retiradas de nota fiscais anonimizadas, constando: a descrição do produto truncada, um código na Nomenclatura Comum do Mercosul (NCM) declarado pelo emissor, e um rótulo anotado por auditores fiscais especialistas. 

In [ ]:
%pip install tensorflow
%pip install itables
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
#@title Funções auxiliares
import os, joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

DIR_ART = 'artefatos'

def _caminho(nome, ext):
    return os.path.join(DIR_ART, f'{nome}.{ext}')

def salva(nome, obj):
    """Matriz esparsa -> .npz; qualquer outro objeto -> .joblib."""
    os.makedirs(DIR_ART, exist_ok=True)
    if sp.issparse(obj):
        sp.save_npz(_caminho(nome, 'npz'), obj.tocsr())
    else:
        joblib.dump(obj, _caminho(nome, 'joblib'), compress=3)

def carrega(nome):
    if os.path.exists(_caminho(nome, 'npz')):
        return sp.load_npz(_caminho(nome, 'npz'))
    return joblib.load(_caminho(nome, 'joblib'))

def existe(nome):
    return any(os.path.exists(_caminho(nome, e)) for e in ('npz', 'joblib'))

def comoDataFrame(M, vocab, indice=None, linhas=slice(0, 20), so_nao_nulas=True):
    """Fatia de M como DataFrame, para visualização. `linhas`: slice ou lista de posições.
    Com so_nao_nulas=True, mostra só as colunas com algum valor nas linhas escolhidas."""
    sub = M[linhas]
    idx = None if indice is None else pd.Index(indice)[linhas]
    cols = np.unique(sub.indices) if so_nao_nulas else np.arange(M.shape[1])
    return pd.DataFrame(sub[:, cols].toarray(), index=idx, columns=np.asarray(vocab)[cols])

from xgboost import XGBClassifier

class XGBClassifierRotulos(XGBClassifier):
    """XGBClassifier que aceita rótulos não contíguos: no CV, classes raras podem faltar no treino do fold,
    e o XGBoost exige rótulos 0..C-1. Renumera no fit e desfaz no predict."""
    def fit(self, X, y, **kw):
        self.rotulos_, y = np.unique(y, return_inverse=True)
        return super().fit(X, y, **kw)
    def predict(self, X, **kw):
        return self.rotulos_[np.asarray(super().predict(X, **kw)).astype(int)]


In [2]:
#@title Carrega o dataset

import os
import pandas as pd
from langdetect import detect, DetectorFactory, LangDetectException
from itables import show
# Carrega a partir do github
base = pd.read_csv('https://github.com/LeoKiepper/Ciencia-de-Dados/raw/main/trabalho2/descmerc/BaseDESC_NCM.zip',
                   sep='\t', dtype={'XPROD': str, 'NCM': str})

show(base)
show(base['NCM'].value_counts())
print("Quantas classes resultam em cada nível do NCM")
print({k: base['NCM'].str[:k].nunique() for k in (2, 4, 6, 8)})


Quantas classes resultam em cada nível do NCM
{2: 82, 4: 703, 6: 1628, 8: 2174}


# Metodogia

### Pré-processamento

Os dados da coluna `NCM` são tratados como labels, e passados por um *label encoder*. A coluna `XPROD` tem dados formatados como *string*. Para tratar a imprevisibilidade do estilo de escrita de cada descrição, foi definida a função de preprocessamento abaixo.

A lógica por trás dessa função é remover ruído das descrições que não contribuiriam para descrever o produto, tomando cuidado para não remover sinal junto com o ruído, levando em conta características específicas de como pessoas escrevem essas descrições. Duas regras globais nortearam a implementação do pré-processamento:

>"Qualificadores que sejam aplicáveis a múltiplos produtos de múltiplas categorias não contribuem para individualizar o produto para uma categoria, então são ruído". (Por exemplo, características da embalagem, marketing como 'leve X pague Y', quantidades)

>"Se há risco de eliminar sinal, não remover um qualificador. A informação passa a ser dado para o algoritimo aprender a usar ou ignorar.".

Um caso notável de qualificadores que foram mantidos pela segunda regra são cores. Para alguns produtos, a cor é um qualificador que não muda a função. Por exemplo, uma escova de dente pode ser de qualquer cor, e ter exatamente a mesma função. Para outros, a cor é um qualificador que distingue o produto de outros semelhantes. Por exemplo, vinho branco é diferente de vinho tinto.

Com base nessas duas regras, foi criada uma sequência de regex para filtrar as strings. Esses regex removem grafias observadas para quantidades (que em geral não seguem grafia padronizada de unidades de medida, como 'metro' → 'm', 'litro' → 'l', 'grama' → 'g, etc). Depois de filtradas, as strings são tokenizadas. Alguns problemas são atacados em mais de uma etapa, para retirá-los de formas diferentes que não são fáceis de pegar por uma regra única. São feitas as operações listadas abaixo. Para listagens completas das chaves regex usadas, vide o código da célula seguinte.

1. Remover acentos e quantidades/tamanhos de embalagem
2. Dividir o texto em tokens
3. Converter o texto para letras minúsculas
4. Separar tokens que usam '.' ou '/' como separador
5. Remover símbolos de pontuação de cada token e descartar tokens vazios
6. Separar palavra colada a uma quantidade (fugini100gr -> fugini, 100gr)
7. Remover unidades, embalagens, marketing, tamanho, referência e stop words
8. Remover letras soltas (tokens com apenas uma letra)
9. Remover números soltos (3, 1/2, 030003, 20x25) e quantidades (750ml, 1kg)
10. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)




In [ ]:
#@title Tokeniza o dataset
import re, unicodedata
import pandas as pd
import nltk
import string
from nltk import word_tokenize
from nltk.corpus import stopwords
from itables import show
from collections import Counter
from functools import lru_cache

nltk.download('punkt_tab')
nltk.download('stopwords')

NUM  = r'\d+(?:[.,]\d+)?'
UNID = r'(?:M2|MLN|ML|MLS|M|MT|MTS|G|GR|GRS|KG|KGS|L|LT|LTS|LI|LITROS?|UN|UND|UNDS|UNID|U|PC|PCS|KIT)'
EMB  = r'(?:CX|CXA|UN|UND|PT|PC|FD|DP|SC|CJ|PCT|BD|FR|GA|GF|DZ)'
FIM  = r'\.?(?![A-Z\d])'          # fim do token: não seguido de letra ou dígito
PADRAO_QTD = re.compile(rf'''
    (?<![\w,.]){EMB}?(?:{NUM}\s?X\s?)+{NUM}\s?{UNID}{FIM}   # 12X500ML, DP12X24X22G
  | (?<![\w,.]){EMB}(?:{NUM}\s?X\s?)+{NUM}{FIM}             # CX20X50
  | (?<![\w,.])1\s?X\s?\d+{FIM}                             # 1 X 12
  | (?<![\w,.]){EMB}?{NUM}\s?{UNID}{FIM}                    # 500ML, 1,5 L, CX10KG
  | \b{EMB}\s?[/-]\s?\d+(?:\s?[/-]\s?{EMB})?{FIM}           # CX/12, UN/0001, CX-6-UN
  | \b{EMB}\d+{EMB}?{FIM}                                   # PT0001UN
  | \bC/\s?\d+(?:\s?{UNID})?{FIM}                           # C/12, C/20MTS
  | \bQTD\.?\s?{NUM}(?:\s{EMB})?                            # QTD. 2.00 DP
  | \(\s?\d+\s?{UNID}?\s?\)                                 # (12 UN)
  | \(CX[\d,]+(?:M2|PCS?)\)                                 # (CX1,62M2)
  | -\s?\d+VL\b                                             # - 24VL
  | \bF\.?P\.?S\.?\s?-?\s?\d+\+?                            # FPS 70
  | \bLV?\d+\s?PG?\d+(?:ML?|G|GR)?\b                        # LV12PG11, L4P3, LV16 PG15
  | \b\d+\s?{EMB}\s?\d+\s?{UNID}{FIM}                       # 1CX12UN
''', re.X)
ALFANUM = re.compile(r'(?=.*\d)(?=.*[a-z])[a-z\d]+')    # tem letra e dígito
FRAG    = re.compile(r'[a-z]{3,}')                      # fragmento alfabético resgatável

# '.' e '/' usados como separador (não quebra decimais e frações: 2.5mm, 1/2)
SEP = re.compile(r'(?<!\d)[./]|[./](?!\d)')
TABELA_PONT = str.maketrans('', '', string.punctuation)
# número+unidade dentro de token (CAB.SAV.750ML -> 750ml, 1-KG -> 1kg, 12x40g)
QTD_TOKEN = re.compile(r'(?:\d+x)*\d+(?:m2|mln|ml|mls|m|mt|mts|g|gr|grs|kg|kgs|l|lt|lts|li|un|und|unid|pc|pcs)')
# palavra colada a uma quantidade (FUGINI100GR -> fugini + 100gr; HARIBO12X350G -> haribo + 12x350g)
COLADO = re.compile(r'([a-z]{2,})((?:\d+x)*\d+(?:m2|mln|ml|mls|m|mt|mts|g|gr|grs|kg|kgs|l|lt|lts|li|un|und|unid|pc|pcs))')

# tokens só de dígitos, separadores e 'x': 3, 1/2, 030003, 20x25
NUMERICO = re.compile(r'[\d.,/+\-%*:;"\'&#x]*\d[\d.,/+\-%*:;"\'&#x]*')
# códigos de referência (preserva DN15, RJ11, IP65, E27, MM2, CA50 e medidas como 2700K, 6800W)
CODIGO = re.compile(r'(?:ref|cod|mva|lote|venc|nr|n\.)[.:\-]?\w[\w.\-/]*'          # REF 123, MVA:29
                    r'|[a-z]-?\d+[a-z]?'                                          # C21, R0091, S08
                    r'|[a-z]-\d+/\d+'                                             # B-4472/17
                    r'|\d{4,}-?(?!(?:k|w|v|a|kv|ka|va|mm|cm|m)$)[a-z]{1,2}'        # 27814E, 1328-M
                    r'|[a-z]{1,2}\d{4,}'                                          # MK2431
                    r'|r\.\d+|a/\d+|tam\d+|\d+h\d+'                               # R.123, A/4552, TAM2, 1H15
                    r'|(?!dn|rj|ip|mm|ca|e\d)[a-z]{2}-?\d+[a-z]?')                # SG-8407, GL83, BR-01
# dimensões NxM(xP), com ou sem unidade: 20x25, 67x370mm, 32mmx22mm, 2750x1840x15mm, 19mmx10m
DIMENSAO = re.compile(r'(?:\d+(?:mm|cm|mts?|m)?x)+\d+(?:mm|cm|mts?|m)?')

def _sem_acento(s): return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()

RUIDO = (
    {'ml','mls','gr','grs','kg','kgs','lt','lts','un','und','unid','cx','cxa','fd','dp','pct','pc','pcs',
     'mt','mts','pt','fr','sc','bd','gf'}                                         # unidades e embalagens
  | {'pack','sache','refil','cartela','pet','lata'}                               # embalagem
  | {'premium','prem','especial','classic','classico','class','super','ultra','plus','max','maxi','top',
     'gold','golden','star','prime','master','power','smart','easy','life','ideal','royal','imperial',
     'new','novo','nova','tradic','orig','duo','one'}                             # marketing / linha
  | {'grande','gde','pequeno','pequena','peq'}                                    # tamanho
  | {'ref','cod','nr'}                                                            # referência
  | ({_sem_acento(w) for w in stopwords.words('portuguese')} - {'com','sem','nao'})
)

@lru_cache(maxsize=None)
def _tokeniza(df: pd.Series):
    # 1. Remover acentos e quantidades/tamanhos de embalagem (sobre a string)
    texto = PADRAO_QTD.sub(' ', _sem_acento(df).upper())

    # 2. Dividir o texto em tokens
    tokens = word_tokenize(texto)

    # 3. Converter o texto para letras minúsculas
    tokens = [w.lower() for w in tokens]

    # 4. Separar tokens que usam '.' ou '/' como separador (VIN.PORT.CHAMINE, P/TRAT)
    tokens = [p for w in tokens for p in SEP.split(w)]

    # 5. Remover símbolos de pontuação de cada token e descartar tokens vazios
    tokens = [w.translate(TABELA_PONT) for w in tokens]
    tokens = [w for w in tokens if w]

    # 6. Separar palavra colada a uma quantidade (fugini100gr -> fugini, 100gr)
    tokens = [p for w in tokens for m in [COLADO.fullmatch(w)] for p in (m.groups() if m else (w,))]

    # 7. Remover unidades, embalagens, marketing, tamanho, referência e stop words
    tokens = [w for w in tokens if w not in RUIDO]

    # 8. Remover letras soltas
    tokens = [w for w in tokens if not (len(w) == 1 and w.isalpha())]

    # 9. Remover números soltos (3, 1/2, 030003), quantidades (750ml, 1kg) e dimensões (67x370mm)
    tokens = [w for w in tokens if not NUMERICO.fullmatch(w)
                               and not QTD_TOKEN.fullmatch(w)
                               and not DIMENSAO.fullmatch(w)]

    # 10. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)
    tokens = [w for w in tokens if not CODIGO.fullmatch(w)]

    return tokens
  
# Em quantas descrições da base cada token aparece (calculado uma vez)
DF_TOKENS = Counter(t for toks in base['XPROD'].map(_tokeniza) for t in set(toks))

def resgata(tokens):
    """Alfanuméricos que aparecem em no máximo uma descrição da base são trocados
    pelos seus fragmentos alfabéticos (>= 3 letras, fora do RUIDO); sem fragmento, somem."""
    out = []
    for w in tokens:
        if DF_TOKENS[w] <= 1 and ALFANUM.fullmatch(w):
            out += [p for p in FRAG.findall(w) if p not in RUIDO]
        else:
            out.append(w)
    return out

def preprocessamento(df: pd.Series):
    # Tokeniza e resgata fragmentos de alfanuméricos únicos
    return resgata(_tokeniza(df))

[nltk_data] Downloading package punkt_tab to /home/Leo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/Leo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#@title Calcula a matriz TF-IDF do dataset completo
import os, io, urllib.request
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

REPO_RAW = 'https://raw.githubusercontent.com/LeoKiepper/Ciencia-de-Dados/main/trabalho2/'
ARQ_X = _caminho('X_tfidf', 'npz')
ARQ_VOCAB = _caminho('X_tfidf_vocab', 'csv')

def calculaX(x: pd.Series):
    """TF-IDF de x. Devolve (M, vocab): M é CSR float32 (linhas na ordem de x), vocab é o array de tokens."""
    vectorizer = TfidfVectorizer(analyzer=preprocessamento, dtype=np.float32)
    M = vectorizer.fit_transform(x)
    return M, vectorizer.get_feature_names_out()

def _le_vocab(origem):
    return pd.read_csv(origem, keep_default_na=False)['token'].to_numpy()

def _baixa_do_github():
    M = sp.load_npz(io.BytesIO(urllib.request.urlopen(REPO_RAW + ARQ_X).read()))
    return M, _le_vocab(REPO_RAW + ARQ_VOCAB)

def _grava(M, vocab):
    os.makedirs(DIR_ART, exist_ok=True)
    sp.save_npz(ARQ_X, M)
    pd.Series(vocab, name='token').to_csv(ARQ_VOCAB, index=False)

def carregaX(x: pd.Series):
    if os.path.exists(ARQ_X) and os.path.exists(ARQ_VOCAB):
        return sp.load_npz(ARQ_X), _le_vocab(ARQ_VOCAB)
    try:
        M, vocab = _baixa_do_github()
    except Exception:
        M, vocab = calculaX(x)
    _grava(M, vocab)
    return M, vocab


### Definição de variáveis de entrada, hiperparâmetros e métricas

O trabalho-referência faz predição sobre a coluna `rotulo` a partir de `XPROD`. Por esse motivo, os resultados trazidos lá não serão diretamente comparáveis. Para obter um termo de comparação, o algoritimo BERT, que lá obteve o melhor resultado, foi aplicado para a predição do `NCM` a partir de `XPROD`. Além disso, foi testado *Random Forest* (RF) como baseline para detecção de sinal, SVM, que até hoje é um algoritimo importante, XGBoost, que está sendo muito usado atualmente na literatura, ===e uma rede neural===.

Para medir o desempenho das predições, foi selecionado o *F1-score*, pois, nessa aplicação, não faz sentido atribuir um peso maior para *precision* ou *recall*. No entanto, para obter uma contextualização mais completa dos resultados dos demais algoritimos contra o BERT e contra o trabalho-referência, é mostrado também a acurácia, pois essa foi a métrica escolhida lá.

Os algoritimos de classificação testados devem exprimir relações entre a descrição do produto (coluna `XPROD` do *dataset*, entrada) e o código `NCM` (saída, ou predição), até os 4 primeiros dígitos. O dataset completo é passado por um split, reservando 25% para teste e 75% para treino, depois são aplicados os respectivos pré-processamentos.

Cada algoritimo de classificação foi treinado com validação cruzada usando a biblioteca GridSearchCV. O grid de hiperparâmetros de cada um dos algoritimos é explicado abaixo;


In [ ]:
#@title Define variáveis de entrada para os algoritimos de classificação
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


USAR_AMOSTRA = False
N_AMOSTRA = 1000
TEST_SIZE = 0.25
NIVEL_NCM = 4
SEMENTE_AMOSTRA = None
SEMENTE_SPLIT = 345510517

if USAR_AMOSTRA:
    semente_amostra = SEMENTE_AMOSTRA if SEMENTE_AMOSTRA is not None else int(np.random.randint(0, 2**31))
    df = base.sample(n=N_AMOSTRA, random_state=semente_amostra)
else:
    semente_amostra = None
    df = base

# linhas de X_full na ordem de df
X_full, vocab = carregaX(base['XPROD'])
X = X_full[base.index.get_indexer(df.index)]
le = LabelEncoder()
y = pd.Series(le.fit_transform(df['NCM'].str[:NIVEL_NCM]), index=df.index, name='NCM')

print(f'X {X.shape} {X.dtype}, {X.nnz} valores não nulos')
show(comoDataFrame(X, vocab, indice=df.index))
print(pd.Series(vocab).sample(50).tolist())


# Separa uma fatia do dataset para teste
semente_split = SEMENTE_SPLIT if SEMENTE_SPLIT is not None else int(np.random.randint(0, 2**31))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=semente_split)

for nome, obj in {'X_train': X_train, 'X_test': X_test,
                  'y_train': y_train, 'y_test': y_test, 'le': le}.items():
    salva(nome, obj)

# Com alguma semente sorteada, os arquivos do split são temporários. a avaliação os remove após gravar o log
salva('config_dados', {'usar_amostra': USAR_AMOSTRA,
                       'n_amostra': len(df),
                       'semente_amostra': semente_amostra,
                       'test_size': TEST_SIZE,
                       'semente_split': semente_split,
                       'guardou_matrizes': not ((USAR_AMOSTRA and SEMENTE_AMOSTRA is None) or SEMENTE_SPLIT is None),
                       'nivel_ncm': NIVEL_NCM
})


X (1117623, 84587) float32, 4937893 valores não nulos


['shsab', '100ml3972005', '300mg150', 'ecologico', 'antiresid', 'x140mmr4851', 'miguelito', 'aralinho', 'tqo', 'hdcvi', 'basso', '045cm', 'edp100', '050mmx90o', 'assentobranco', 'semibrilho', 'lysorform', 'taylorsporto', '19w', 'moranguete', 'xim', '2005v2', 'eyebrown', 'bloqueio', 'bonde', 'nata', 'cuador', 'culinrio', 'kst', 'menthe', 'printemps', 'craker', '04000630a', 'brooklyn', 'cest', 'unhas', 'lineas', 'restaurador', 'tud223', 'atomic', 'provola', 'csnr', 'scuderia', 'sgr', 'cereal', 'tigre60mm', 'malli', 'istantaneo', 'escura', 'f26x150mx50m']


In [2]:
#@title Suite de treino: GridSearchCV por partes (salvas uma a uma) -> melhor candidato -> refit
import os, glob, json, time, datetime, joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV

IGNORA_HASH = {'n_jobs', 'verbose'}          # parâmetros que não mudam o resultado

def f1_macro_em_lotes(estimador, X, y, lote=2000):
    """macro-F1 com predict em lotes: o predict_proba da RF aloca threads × linhas × classes em float64."""
    yhat = np.concatenate([estimador.predict(X[i:i + lote]) for i in range(0, X.shape[0], lote)])
    return f1_score(y, yhat, average='macro')

def _json(pasta, info=None):
    """Lê (info=None) ou grava o descricao.json da pasta."""
    arq = os.path.join(DIR_ART, pasta, 'descricao.json')
    if info is None:
        with open(arq) as f: return json.load(f)
    with open(arq, 'w') as f: json.dump(info, f, indent=2, ensure_ascii=False, default=repr)

def _acha_pasta(h):
    """Pasta grid_* cujo descricao.json tem o hash h (ou None)."""
    for arq in sorted(glob.glob(os.path.join(DIR_ART, 'grid_*', 'descricao.json'))):
        pasta = os.path.basename(os.path.dirname(arq))
        if _json(pasta)['hash'] == h:
            return pasta
    return None

def _treina(nome, estimador, grid, X_train, y_train, cv=5, scoring=f1_macro_em_lotes,
            n_jobs=1, verbose=3, refaz=False):
    """grid: dict ou lista de dicts; cada dict é uma parte (um GridSearchCV), salva assim que termina.
    Depois junta as partes, escolhe o melhor e retreina em todo X_train."""
    grids = grid if isinstance(grid, list) else [grid]
    desc = {'estimador': f'{type(estimador).__module__}.{type(estimador).__name__}',
            'parametros': {k: v for k, v in estimador.get_params(deep=False).items() if k not in IGNORA_HASH},
            'grid': grids}
    h = joblib.hash(desc)

    pasta = None if refaz else _acha_pasta(h)
    if pasta is None:
        agora = datetime.datetime.now()
        pasta = f'grid_{agora:%Y%m%d_%H%M%S}_{nome}'
        os.makedirs(os.path.join(DIR_ART, pasta))
        _json(pasta, {'nome': nome, 'hash': h, **desc, 'criada': f'{agora:%Y-%m-%d %H:%M:%S}', 'concluido': False})
    elif (_json(pasta)['concluido'] and existe(f'modelo_{nome}') and existe(f'treino_{nome}')
          and carrega(f'treino_{nome}').get('hash') == h):
        print(f'[{nome}] já treinado com este grid ({pasta}, hash {h[:8]}), pulando')
        return nome
    print(f'[{nome}] partes em {DIR_ART}/{pasta}/ (hash {h[:8]}): {len(grids)} partes × {cv} folds')

    partes = []
    for i, g in enumerate(grids, 1):
        arq = f'{pasta}/parte_{i:03d}'
        if existe(arq):
            partes.append(carrega(arq)); print(f'[{nome} {i}/{len(grids)}] já salva, pulando'); continue
        busca = GridSearchCV(clone(estimador), g, cv=cv, scoring=scoring,
                             n_jobs=n_jobs, refit=False, verbose=verbose)
        t = time.perf_counter(); busca.fit(X_train, y_train); t = time.perf_counter() - t
        p = {'grid': g, 'cv_results': pd.DataFrame(busca.cv_results_), 'tempo_s': t}
        salva(arq, p); partes.append(p)
        print(f'[{nome} {i}/{len(grids)}] {t / 60:.1f} min; melhor {p["cv_results"].mean_test_score.max():.4f}')

    cv_results = pd.concat([p['cv_results'] for p in partes], ignore_index=True)
    cv_results['rank_test_score'] = cv_results.mean_test_score.rank(ascending=False, method='min').astype(int)
    melhor = cv_results.loc[cv_results.mean_test_score.idxmax()]
    t = time.perf_counter()
    modelo = clone(estimador).set_params(**melhor['params']).fit(X_train, y_train)
    t_refit = time.perf_counter() - t
    tempo = sum(p['tempo_s'] for p in partes) + t_refit
    salva(f'modelo_{nome}', modelo)
    salva(f'treino_{nome}', {
        'melhores_parametros': melhor['params'],
        'parametros': modelo.get_params(),
        'melhor_score_cv': melhor['mean_test_score'],
        'cv_results': cv_results,                       # todas as partes juntas, para plots
        'tempo_s': tempo,
        'indice_treino': y_train.index,                 # auditoria
        'pasta_partes': pasta,
        'hash': h,
    })
    _json(pasta, {**_json(pasta), 'concluido': True, 'tempo_s': tempo,
                  'melhor_score_cv': float(melhor['mean_test_score']), 'melhores_parametros': melhor['params']})
    print(f'[{nome}] refit {t_refit / 60:.1f} min; melhores: {melhor["params"]}')
    del modelo                                          # libera a RAM antes do próximo algoritmo
    return nome

def treina(modelos, X_train, y_train, **opcoes):
    """modelos: lista de (nome, estimador, grid). Devolve a lista de nomes."""
    return [_treina(nome, est, grid, X_train, y_train, **opcoes) for nome, est, grid in modelos]


In [4]:
#@title Treina os algoritmos
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Random Forest: pares
RF_PRODUTO_TL = 600_000
RF_L = np.geomspace(4000, 150000, 16).round(-1).astype(int)
RF_PARES = sorted({(int(max(1, RF_PRODUTO_TL // L)), int(L)) for L in RF_L})
RF_BASE = {'max_features': ['sqrt'], 'class_weight': [None, 'balanced_subsample'], 'bootstrap': [True, False]}

# # XGBoost: PROVISÓRIO, a calibrar no ensaio (tempo por rodada, CPU vs GPU, pico de RAM/VRAM)
# XGB_DEVICE = 'cuda'                   # 'cuda' ou 'cpu', definido no ensaio
# XGB_PARES = [(200, 0.3), (400, 0.15)] # (n_estimators, learning_rate): mesmo "orçamento" de aprendizado
# XGB_BASE = {'max_depth': [6], 'subsample': [0.8], 'colsample_bytree': [0.8]}

MODELOS = [
    ('RandomForestClassifier', RandomForestClassifier(n_jobs=20, random_state=0),
     [{**RF_BASE, 'n_estimators': [T], 'max_leaf_nodes': [L]} for T, L in RF_PARES]),
    # ('XGBClassifier', XGBClassifierRotulos(tree_method='hist', device=XGB_DEVICE, n_jobs=20, random_state=0),
    #  [{**XGB_BASE, 'n_estimators': [n], 'learning_rate': [lr]} for n, lr in XGB_PARES]),
]

X_train, y_train = carrega('X_train'), carrega('y_train')
treina(MODELOS, X_train, y_train)



[RandomForestClassifier] já treinado com este grid (grid_20261118_164829, hash 99c9f726), pulando


['RandomForestClassifier']

In [5]:
#@title Suite de inferência
import numpy as np

def infere(modelo, X, lote=2000):
    return np.concatenate([modelo.predict(X[i:i + lote]) for i in range(0, X.shape[0], lote)])

In [6]:
#@title Inferência: ŷ_test = A(X_test)
MODELOS_INFERIDOS = ['RandomForestClassifier']

X_test = carrega('X_test')
for nome in MODELOS_INFERIDOS:
    if not existe(f'yhat_test_{nome}'):
        salva(f'yhat_test_{nome}', infere(carrega(f'modelo_{nome}'), X_test))


In [11]:
#@title Suite de avaliação
import os, sys, hashlib
import numpy as np, scipy, sklearn
import pandas as pd
from sklearn.metrics import classification_report

def _medidas(y_true, y_pred):
    return {
        'classification_report': classification_report(y_true, y_pred, output_dict=True, zero_division=0),
    }

def _assegura_sem_vazamento(nome, treino, y_true):
    vazadas = treino['indice_treino'].intersection(y_true.index)
    assert len(vazadas) == 0, f'[{nome}] {len(vazadas)} linhas avaliadas foram usadas no treino'

def _relatorio_do_modelo(nome, treino, y_true, y_pred):
    _assegura_sem_vazamento(nome, treino, y_true)
    relatorio = _medidas(y_true, y_pred)['classification_report']
    return [f'--- {nome} ---',
            f'Melhores parâmetros (CV): {treino["melhores_parametros"]}',
            f'Score médio na CV: {treino["melhor_score_cv"]:.4f}; tempo de treino: {treino["tempo_s"]:.0f} s',
            f'Acurácia {relatorio["accuracy"]:.4f} | macro-F1 {relatorio["macro avg"]["f1-score"]:.4f} '
            f'(precision {relatorio["macro avg"]["precision"]:.4f}, recall {relatorio["macro avg"]["recall"]:.4f})',
            f'Parâmetros completos: {treino["parametros"]}',
            pd.DataFrame(relatorio).T.to_string(), '']

def sha256_do_arquivo(caminho):
    with open(caminho, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

def cabecalho(agora, config, arq_x, sha_x):
    return '\n'.join([f'=== Avaliação {agora:%Y-%m-%d %H:%M:%S} ===',
                      f'Dados: {config}',
                      f'Matriz TF-IDF: {arq_x} sha256={sha_x}',
                      f'Versões: python {sys.version.split()[0]}, numpy {np.__version__}, '
                      f'scipy {scipy.__version__}, scikit-learn {sklearn.__version__}', '', ''])

def avalia(treinos, y_true, yhats):
    linhas = []
    for nome, treino in treinos.items():
        linhas += _relatorio_do_modelo(nome, treino, y_true, yhats[nome])
    return '\n'.join(linhas)

def remove_split_temporario(config):
    if config['guardou_matrizes']:
        return
    for nome in ('X_train', 'X_test', 'y_train', 'y_test', 'config_dados'):
        for ext in ('npz', 'joblib'):
            if os.path.exists(_caminho(nome, ext)):
                os.remove(_caminho(nome, ext))
    print('Arquivos temporários do split removidos; sementes registradas no log')


In [ ]:
#@title Avalia: y_test vs ŷ_test
import datetime

MODELOS_AVALIADOS = ['RandomForestClassifier']

config = carrega('config_dados')
y_test = carrega('y_test')
treinos = {nome: carrega(f'treino_{nome}') for nome in MODELOS_AVALIADOS}
yhats_test = {nome: carrega(f'yhat_test_{nome}') for nome in MODELOS_AVALIADOS}

agora = datetime.datetime.now()
ARQ_X_TFIDF = _caminho('X_tfidf', 'npz')
texto = cabecalho(agora, config, ARQ_X_TFIDF, sha256_do_arquivo(ARQ_X_TFIDF)) + avalia(treinos, y_test, yhats_test)
print(texto)
with open(_caminho(f'avaliacao_{agora:%Y%m%d_%H%M%S}', 'log'), 'w') as f:
    f.write(texto + '\n')
remove_split_temporario(config)


=== Avaliação 2026-11-19 17:13:19 ===
Dados: {'usar_amostra': False, 'n_amostra': 1117623, 'semente_amostra': None, 'test_size': 0.25, 'semente_split': 345510517, 'guardou_matrizes': True, 'nivel_ncm': 4}
Matriz TF-IDF: X_tfidf.npz sha256=bef63211aea005880f69c96490f63c20cd61fd694f1b10723976c09453b24969
Versões: python 3.13.18, numpy 2.4.6, scipy 1.17.1, scikit-learn 1.9.1

--- RandomForestClassifier ---
Melhores parâmetros (CV): {'bootstrap': False, 'class_weight': None, 'max_features': 'sqrt', 'max_leaf_nodes': 150000, 'n_estimators': np.int64(4)}
Score médio na CV: 0.5048; tempo de treino: 85954 s
Acurácia 0.9046 | macro-F1 0.5075 (precision 0.5587, recall 0.4896)
Parâmetros completos: {'bootstrap': False, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': 150000, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None